# Emotion-aware speech2face — live streaming with the trained emotion adapter

Runs the same real-time PersonaPlex → Helium-student → distilled-renderer server as
`runpod_speech2face.ipynb`, with the **emotion adapter** from
`train_emotion_lora.ipynb` merged into the motion student. Robert still speaks as Robert —
only the **face** expresses the selected emotion (exactly like the old pipeline's
`06_emotion_live_pipeline.ipynb`).

- **Startup emotion** via the `EMOTION` config below (env `EMOTION_ID`).
- **Runtime switching**: writing an emotion id or name to `emotion_control.txt` switches
  the avatar mid-conversation (an in-place buffer update — the student's CUDA graph keeps
  running, no restart, no re-capture). Section 8 has a one-line helper.
- Default port **8992** — but do **not** run both servers at once (two PersonaPlex
  instances ≈ 28 GB of VRAM).

**Prerequisites**
1. the base setup on this pod (`weights/`, `.python_path`, `.gpu_probe.env`) — either
   `runpod_speech2face.ipynb` or the **Section 0** cells of `train_emotion_lora.ipynb`
   (the preflight audit checks all of them and a repair cell covers the public weights).
2. A trained adapter: `train_emotion_lora.ipynb` on this pod, **or** an adapter on
   Hugging Face (`EMOTION_HF_REPO` below).

Nothing existing is modified: this notebook writes three **new** files
(`src/niloys/emotion/emotion_student.py`,
`src/niloys/live/imtalker_emotion_student_live.py`, `run_emotion_x86.sh`).

In [ ]:
import os, re, json, glob, shlex, subprocess, time, urllib.request
from pathlib import Path

REPO_DIR = "/workspace/speech2face"
PORT = 8992                        # 8998 is refused by the server; 8991 may hold the base one

# ---- Python interpreter of the pod env (same resolution as run_x86.sh):
#      PY_OVERRIDE -> .python_path (written by prepare_x86.sh) -> env/bin/python -> python3
PY_OVERRIDE = ""                   # e.g. "/workspace/speech2face/env/bin/python"

def _resolve_python():
    pp = os.path.join(REPO_DIR, ".python_path")
    for cand in (PY_OVERRIDE,
                 open(pp).read().strip() if os.path.exists(pp) else "",
                 os.path.join(REPO_DIR, "env", "bin", "python")):
        if cand and os.path.exists(cand):
            return cand
    return "python3"

PY = _resolve_python()

def audit_base(include_live=False):
    # every base-setup artifact the emotion pipeline needs:
    # returns rows of (group, path, ok, repairable_in_notebook)
    W = os.path.join(REPO_DIR, "weights")
    rows = []
    def add(group, path, repairable):
        rows.append((group, path, os.path.exists(path), repairable))
    pp = os.path.join(REPO_DIR, ".snap_path")
    snap = open(pp).read().strip() if os.path.exists(pp) else ""
    add("base model", pp, False)
    if snap:
        add("base model", os.path.join(snap, "model.safetensors"), False)
    add("base model", os.path.join(W, "mimi.safetensors"), False)
    add("base model", os.path.join(W, "tokenizer_spm_32k_3.model"), False)
    st = os.path.join(W, "motionM.pt")
    if not os.path.exists(st):
        st = os.path.join(W, "motion_student_38M.pt")
    add("avatar assets", st, True)
    for f in ("renderer_narrow.pt", "silence_helium_mean.pt",
              "blink_motion.pt", "ref.jpeg"):
        add("avatar assets", os.path.join(W, f), True)
    add("avatar assets", os.path.join(W, "voices", VOICE), True)
    add("gpu probe", os.path.join(REPO_DIR, ".gpu_probe.env"), True)
    if include_live:
        for f in ("pd_quant_onpol_robert5_plaus2.pt", "mimi_dec_gan_robert5.pt"):
            add("robert5 gan", os.path.join(W, f), True)
    return rows

def print_audit(rows):
    print("%-14s %-8s %s" % ("group", "status", "artifact"))
    for group, path, ok, _ in rows:
        print("%-14s %-8s %s" % (group, "OK" if ok else "MISSING", path))

EMOTION = "happy"                  # startup emotion: neutral happy sad angry surprise fear (or 0-5)
EMOTION_LORA_CKPT = "/workspace/emotion_dataset/emotion_lora_v1/checkpoints/last.pt"
EMOTION_HF_REPO = ""               # optional: "user/repo" to download the adapter instead
EMOTION_HF_FILE = "last.pt"        # filename inside that HF repo

VOICE = "Robert_5.pt"
TEXT_PROMPT = ("You are Robert, an assistant from RB Labs. You have knowledge about crypto, "
               "finance, investment, technology, and general topics. Answer every question "
               "completely, descriptively, and with useful detail.")
GAIN = "1.2"
PREBUFFER = "3"

def sh(cmd, cwd=None, check=True, env=None, quiet=False):
    """Run a shell command, streaming its output into the notebook as it happens."""
    cwd = cwd or (REPO_DIR if os.path.isdir(REPO_DIR) else os.getcwd())
    if not quiet:
        print("$ " + cmd, flush=True)
    p = subprocess.Popen(["bash", "-c", cmd], cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1,
                         errors="replace", env={**os.environ, **(env or {})})
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError("command failed (exit %d): %s" % (rc, cmd))
    return rc

print("repo    :", REPO_DIR)
print("python  :", PY)
print("port    :", PORT)
print("emotion :", EMOTION)
print("adapter :", EMOTION_LORA_CKPT)

## 1 · Preflight

Verifies the base setup, resolves the adapter (downloading it from Hugging Face if
`EMOTION_HF_REPO` is set), and stops any already-running server — a base server on 8991
holds ~14 GB of VRAM and two PersonaPlex instances rarely fit on one card.

If the audit flags missing **avatar assets** / **robert5 gan** / **gpu probe** items,
run the repair cell (1b) - it re-downloads the public, sha256-checked weights
(no token needed).

In [ ]:
assert os.path.isdir(REPO_DIR), ("base setup missing - run the Section 0 cells of "
                                 "train_emotion_lora.ipynb (or runpod_speech2face.ipynb)")
assert os.path.exists(PY) or subprocess.run(
    ["bash", "-c", "command -v %s" % shlex.quote(PY)],
    capture_output=True).returncode == 0, (
    "python interpreter not found: %r - run the base setup first (Section 0 of "
    "train_emotion_lora.ipynb), or set PY_OVERRIDE in the config cell" % PY)
sh("%s -c 'import sys, torch; print(\"python:\", sys.executable, \"| torch:\", torch.__version__)'"
   % shlex.quote(PY))
rows = audit_base(include_live=True)
print_audit(rows)
fatal = [r for r in rows if not r[2] and not r[3]]
repair = [r for r in rows if not r[2] and r[3]]
if fatal or repair:
    sh("ls -la %s %s" % (shlex.quote(os.path.join(REPO_DIR, "weights")),
                         shlex.quote(os.path.join(REPO_DIR, "assets"))), check=False)
assert not fatal, (
    "base-model artifacts missing - run cell 0e of train_emotion_lora.ipynb "
    "(or runpod_speech2face.ipynb; HF token + ~16 GB): " + ", ".join(r[1] for r in fatal))
assert not repair, (
    "run the NEXT cell (repair): it downloads the public avatar assets + "
    "robert5_gan weights and probes the GPU. Missing: "
    + ", ".join(os.path.basename(r[1]) for r in repair))
engine = os.path.join(REPO_DIR, "weights", "seanet_w12_fp32.engine")
print("trt engine:", engine if os.path.exists(engine)
      else "absent - the server will use the PyTorch SEANet decoder (a few ms slower)")
student = None
for cand in ("motionM.pt", "motion_student_38M.pt"):
    if os.path.exists(os.path.join(REPO_DIR, "weights", cand)):
        student = cand
        break
print("student :", student)

if not os.path.exists(EMOTION_LORA_CKPT) and EMOTION_HF_REPO:
    os.makedirs(os.path.dirname(EMOTION_LORA_CKPT), exist_ok=True)
    sh("%s -c \"from huggingface_hub import hf_hub_download; import shutil; "
       "p = hf_hub_download('%s', '%s'); shutil.copy(p, '%s'); print('downloaded', p)\""
       % (shlex.quote(PY), EMOTION_HF_REPO, EMOTION_HF_FILE, EMOTION_LORA_CKPT))
assert os.path.exists(EMOTION_LORA_CKPT), (
    "adapter not found: %s\nrun train_emotion_lora.ipynb first, or set EMOTION_HF_REPO"
    % EMOTION_LORA_CKPT)
print("adapter :", EMOTION_LORA_CKPT)

sh("nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader", check=False)
sh("pkill -f run_watchdog_x86.sh; pkill -f imtalker_try19_student_live.py; "
   "pkill -f imtalker_emotion_student_live.py; sleep 2", check=False)
print("preflight ok")

## 1b - Repair - only if the audit flagged avatar-assets / robert5-gan / gpu-probe items

Same re-download as the training notebook's repair cell, plus the public `robert5_gan`
weights the live server needs (the try19 PD + GAN Mimi decoder hooks, ~2.9 GB). Gated
base-model artifacts are not touched - run cell 0e of `train_emotion_lora.ipynb` (or
`runpod_speech2face.ipynb`) for those. Re-run preflight afterwards.

In [ ]:
rows = audit_base(include_live=True)
fatal = [r for r in rows if not r[2] and not r[3]]
missing = [r for r in rows if not r[2] and r[3]]
assert not fatal, ("base-model artifacts missing - run the Section 0 cells of "
                   "train_emotion_lora.ipynb (or runpod_speech2face.ipynb) "
                   "first: " + ", ".join(r[1] for r in fatal))
if not missing:
    print("nothing to repair - all base artifacts present")
else:
    print("repairing:", ", ".join(os.path.basename(r[1]) for r in missing))
    FETCH = ("import hashlib, os, sys\n"
             "from huggingface_hub import hf_hub_download\n"
             "REPO_ID = 'niloy629/personaplex-parallel-depformer'\n"
             "folder, dest, files = sys.argv[1], sys.argv[2], sys.argv[3:]\n"
             "want = {}\n"
             "for line in open(hf_hub_download(REPO_ID, folder + '/SHA256SUMS')):\n"
             "    h, f = line.split(); want[os.path.basename(f)] = h\n"
             "os.makedirs(dest, exist_ok=True)\n"
             "for f in files:\n"
             "    src = hf_hub_download(REPO_ID, folder + '/' + f)\n"
             "    h = hashlib.sha256()\n"
             "    with open(src, 'rb') as fh:\n"
             "        for chunk in iter(lambda: fh.read(1 << 24), bytes()):\n"
             "            h.update(chunk)\n"
             "    if h.hexdigest() != want[f]:\n"
             "        raise SystemExit('hash mismatch for ' + f)\n"
             "    dst = os.path.join(dest, f)\n"
             "    if os.path.islink(dst) or os.path.exists(dst):\n"
             "        os.remove(dst)\n"
             "    os.symlink(src, dst)\n"
             "    print('  fetched %s -> %s' % (f, src), flush=True)\n")
    fp = os.path.join(REPO_DIR, "_fetch_assets.py")
    open(fp, "w").write(FETCH)
    W = os.path.join(REPO_DIR, "weights")
    ASSETS = os.path.join(REPO_DIR, "assets")
    HF_ENV = {"HF_HOME": os.path.join(REPO_DIR, "hf"),
              "HF_HUB_ENABLE_HF_TRANSFER": "0"}
    AV = ["motion_student_38M.pt", "renderer_narrow.pt", "silence_helium_mean.pt",
          "blink_motion.pt", "ref.jpeg", "Robert_5.pt"]
    def _have(f):
        p = os.path.join(W, "voices", f) if f == "Robert_5.pt" else os.path.join(W, f)
        return os.path.exists(p)
    need = [f for f in AV if not _have(f)]
    if not os.path.exists(os.path.join(ASSETS, "seanet_w12.onnx")):
        need.append("seanet_w12.onnx")
    if need:
        sh("%s %s avatar_assets %s %s"
           % (shlex.quote(PY), shlex.quote(fp), shlex.quote(ASSETS), " ".join(need)),
           env=HF_ENV)
        for f in need:
            if f == "seanet_w12.onnx":
                continue
            dst = os.path.join(W, "voices", f) if f == "Robert_5.pt" else os.path.join(W, f)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            if os.path.islink(dst) or os.path.exists(dst):
                os.remove(dst)
            os.symlink(os.path.join(ASSETS, f), dst)
            print("linked %s" % dst)
    gan = [f for f in ("pd_quant_onpol_robert5_plaus2.pt", "mimi_dec_gan_robert5.pt")
           if not os.path.exists(os.path.join(W, f))]
    if gan:
        sh("%s %s robert5_gan %s %s"
           % (shlex.quote(PY), shlex.quote(fp), shlex.quote(W), " ".join(gan)),
           env=HF_ENV)
    if not os.path.exists(os.path.join(REPO_DIR, ".gpu_probe.env")):
        sh("%s tools/gpu_probe.py .gpu_probe.env" % shlex.quote(PY))
    os.remove(fp)
    rows = audit_base(include_live=True)
    print_audit(rows)
    left = [r for r in rows if not r[2]]
    assert not left, "still missing after repair: " + ", ".join(r[1] for r in left)
    print("repair complete - all base artifacts present")

## 2 · The shared emotion module

The identical `src/niloys/emotion/emotion_student.py` the training notebook writes —
re-written here so this notebook also works on a freshly re-cloned repo.

In [ ]:
import os
CONTENT = '"""Emotion conditioning for the Helium motion student (training + live).\n\nFaithful port of the old FM pipeline\'s emotion mechanism\n(IMTalker generator/FM.py::EmotionConditioner + generator/train_lora.py)\nre-targeted at the distilled CausalMotionStudent (src/niloys/student/model.py):\n\n  * StudentEmotionConditioner  one-hot(7) -> Linear(7 -> d_model) -> LayerNorm\n                               -> SiLU, added to the student\'s embedded token\n                               stream. 10% of training steps get a zeroed bias\n                               (classifier-free dropout), like the old pipeline.\n  * EmotionMotionStudent       CausalMotionStudent subclass. The live bias\n                               lives in a registered buffer updated IN PLACE by\n                               set_emotion(), so the captured CUDA graph keeps\n                               reading the same storage when the emotion is\n                               switched mid-conversation.\n  * LoRA helpers               adapters on the transformer blocks (attn.qkv,\n                               attn.proj, mlp.0, mlp.3) and the output heads\n                               (head_abs, head_res, head_gate). The Helium input\n                               path (helium_ln / helium_proj / prev_enc /\n                               fuse_*) is never touched - the same lip-sync\n                               guarantee as freezing the old audio projections.\n\nOnly NEW files live under src/niloys/emotion/ - nothing existing is modified.\n"""\nfrom __future__ import annotations\n\nimport re\n\nimport torch\nimport torch.nn as nn\n\nfrom student.model import CausalMotionStudent, StudentConfig\n\nEMOTION_MAP = {\n    "neutral": 0, "calm": 0,\n    "happy": 1, "happiness": 1, "joy": 1, "amused": 1,\n    "sad": 2, "sadness": 2,\n    "angry": 3, "anger": 3, "rage": 3,\n    "surprise": 4, "surprised": 4,\n    "fear": 5, "fearful": 5, "afraid": 5,\n}\nEMOTION_NAMES = ["neutral", "happy", "sad", "angry", "surprise", "fear"]\nNUM_EMOTIONS = 6\nNUM_CLASSES = NUM_EMOTIONS + 1      # one-hot width incl. the classifier-free null slot\nADAPTER_FORMAT = "student_emotion_adapter_v1"\n\n\ndef parse_emotion_from_stem(stem: str):\n    """MEAD-style names: M003_happy_level_1_001, W011_sad_3_012, ..."""\n    for part in re.split(r"[_\\-\\.\\s]+", stem.lower()):\n        if part in EMOTION_MAP:\n            return EMOTION_MAP[part]\n    return None\n\n\nclass StudentEmotionConditioner(nn.Module):\n    """emotion id -> [B,T,d_model] additive bias (the old EmotionConditioner at d_model=512)."""\n\n    def __init__(self, d_model: int, dropout: float = 0.1):\n        super().__init__()\n        self.dropout = float(dropout)\n        self.emotion_embedding = nn.Embedding(NUM_CLASSES, NUM_CLASSES)\n        with torch.no_grad():\n            self.emotion_embedding.weight.copy_(torch.eye(NUM_CLASSES))\n        self.emotion_embedding.weight.requires_grad_(False)\n        self.emotion_projection = nn.Linear(NUM_CLASSES, d_model)\n        self.layer_norm = nn.LayerNorm(d_model)\n        self.activation = nn.SiLU()\n\n    def forward(self, emotion_ids, batch_size: int, seq_len: int, train: bool = False):\n        emb = self.emotion_embedding(emotion_ids)           # [B,7] one-hot\n        emb = self.emotion_projection(emb)                  # [B,C]\n        emb = self.layer_norm(emb)\n        emb = self.activation(emb)\n        emb = emb.unsqueeze(1).expand(batch_size, seq_len, emb.shape[-1])\n        if train and self.dropout > 0:\n            mask = torch.rand(batch_size, 1, 1, device=emb.device) < self.dropout\n            emb = torch.where(mask, torch.zeros_like(emb), emb)\n        return emb\n\n    @torch.no_grad()\n    def bias_for(self, idx: int, device, dtype):\n        ids = torch.tensor([int(idx)], device=device, dtype=torch.long)\n        return self.forward(ids, 1, 1, train=False).to(device=device, dtype=dtype)\n\nclass EmotionMotionStudent(CausalMotionStudent):\n    """CausalMotionStudent + emotion bias added after _embed()."""\n\n    def __init__(self, cfg: StudentConfig, emotion_dropout: float = 0.1):\n        super().__init__(cfg)\n        self.emotion_conditioner = StudentEmotionConditioner(cfg.d_model, emotion_dropout)\n        self.register_buffer("emotion_bias", torch.zeros(1, 1, cfg.d_model),\n                             persistent=False)\n        self._emotion_bias = None       # streamed bias (the buffer) while live\n\n    # -- live --------------------------------------------------------------\n    @torch.no_grad()\n    def set_emotion(self, idx: int):\n        emb = self.emotion_conditioner.bias_for(\n            idx, self.emotion_bias.device, self.emotion_bias.dtype)\n        self.emotion_bias.copy_(emb)    # in place: the captured CUDA graph keeps working\n        self._emotion_bias = self.emotion_bias\n\n    # -- shared ------------------------------------------------------------\n    def _embed(self, helium, prev_motion):\n        x = super()._embed(helium, prev_motion)\n        if self._emotion_bias is not None:\n            x = x + self._emotion_bias\n        return x\n\n    # -- training ----------------------------------------------------------\n    def forward_with_emotion(self, helium, prev_motion, emotion_ids, train: bool = True):\n        B, T = helium.shape[0], helium.shape[1]\n        bias = self.emotion_conditioner(emotion_ids, B, T, train=train)\n        prev = self._emotion_bias\n        self._emotion_bias = bias\n        try:\n            return super().forward(helium, prev_motion)\n        finally:\n            self._emotion_bias = prev\n\n\n# ---------------------------------------------------------------- LoRA\nLORA_TARGETS = ("attn.qkv", "attn.proj", "mlp.0", "mlp.3",\n                "head_abs", "head_res", "head_gate")\n\n\ndef apply_emotion_lora(model, rank: int = 32, alpha: float = 64.0, dropout: float = 0.05):\n    """Freeze everything, unfreeze the conditioner, wrap LoRA around the blocks\'\n    attention/MLP projections and the output heads. Returns #wrapped layers."""\n    from generator.train_lora import LoRALinear      # needs src/imtalker on sys.path\n    for p in model.parameters():\n        p.requires_grad_(False)\n    for p in model.emotion_conditioner.parameters():\n        p.requires_grad_(True)\n    model.emotion_conditioner.emotion_embedding.weight.requires_grad_(False)\n    n = 0\n    for name, mod in list(model.named_modules()):\n        if not isinstance(mod, nn.Linear):\n            continue\n        if not any(name == t or name.endswith("." + t) for t in LORA_TARGETS):\n            continue\n        parent_name, _, attr = name.rpartition(".")\n        parent = model.get_submodule(parent_name) if parent_name else model\n        setattr(parent, attr, LoRALinear(mod, rank=rank, alpha=alpha, dropout=dropout))\n        n += 1\n    return n\n\n\ndef save_emotion_adapter(model, path, meta):\n    from generator.train_lora import LoRALinear\n    sd = {"format": ADAPTER_FORMAT, "meta": meta,\n          "conditioner": {k: v.detach().cpu()\n                          for k, v in model.emotion_conditioner.state_dict().items()},\n          "lora": {}}\n    for name, mod in model.named_modules():\n        if isinstance(mod, LoRALinear):\n            sd["lora"][name] = {"A": mod.lora_A.weight.detach().cpu(),\n                                "B": mod.lora_B.weight.detach().cpu()}\n    torch.save(sd, path)\n    return sd\n\n\ndef load_base_student(base_ckpt: str):\n    """Fresh trainable copy of the base student (weights + stats buffers ride along)."""\n    pack = torch.load(base_ckpt, map_location="cpu", weights_only=False)\n    cfg = StudentConfig(**pack["cfg"])\n    model = EmotionMotionStudent(cfg)\n    model.load_state_dict(pack["model"], strict=False)   # conditioner keys are new\n    return model, cfg, pack\n\n\ndef build_emotion_student(base_ckpt: str, adapter_ckpt: str, device,\n                          emotion_dropout: float = 0.0):\n    """Base student + merged emotion LoRA + loaded conditioner (eval, frozen).\n\n    The LoRA deltas are merged straight into the base Linear weights - the live\n    path keeps no LoRA wrappers, so the student\'s CUDA graph is unaffected.\n    """\n    model, cfg, _pack = load_base_student(base_ckpt)\n    ad = torch.load(adapter_ckpt, map_location="cpu", weights_only=False)\n    assert ad.get("format") == ADAPTER_FORMAT, \\\n        "not a student emotion adapter: %s" % adapter_ckpt\n    meta = ad.get("meta", {})\n    scale = float(meta.get("lora_alpha", 64.0)) / float(meta.get("lora_rank", 32))\n    for name, d in ad["lora"].items():\n        mod = model.get_submodule(name)\n        mod.weight.data.add_(scale * (d["B"] @ d["A"]).to(mod.weight.dtype))\n    model.emotion_conditioner.load_state_dict(ad["conditioner"])\n    model.emotion_conditioner.dropout = float(emotion_dropout)\n    model.to(device).float().eval()\n    for p in model.parameters():\n        p.requires_grad_(False)\n    return model, cfg, meta\n'
path = os.path.join(REPO_DIR, "src", "niloys", "emotion", "emotion_student.py")
os.makedirs(os.path.dirname(path), exist_ok=True)
with open(path, "w") as f:
    f.write(CONTENT)
print("wrote", path, "(%d chars)" % len(CONTENT))

## 3 · The live wrapper

`imtalker_emotion_student_live.py` imports `imtalker_try19_student_live` as a module and
patches two of its globals before `main()` runs — so the int4 trunk, one-pass depformer,
distilled renderer, TensorRT/GAN Mimi decoder and the CUDA-graphed student stepping are
reused **verbatim**:

- `load_student` → base student + emotion conditioner + LoRA merged into the weights
  (no wrappers at runtime), then `set_emotion(EMOTION_ID)`.
- `make_student_sampler` → the stock sampler, wrapped so each 80 ms chunk first polls
  `emotion_control.txt`; a change triggers an in-place `emotion_bias` update, which the
  captured CUDA graph keeps reading.

In [ ]:
import os
CONTENT = '"""Emotion-aware variant of imtalker_try19_student_live.py.\n\nEvery piece of the verified try19 server (int4 trunk, one-pass depformer,\ndistilled renderer, TensorRT / GAN Mimi decoder, CUDA-graphed student stepping)\nis reused untouched. The only changes, applied by monkey-patching try19\'s\nmodule globals before try19.main() runs:\n\n  * load_student        -> loads the base student, attaches the emotion\n                           conditioner, merges the emotion LoRA adapter into\n                           the base weights, and sets the startup emotion\n                           (EMOTION_ID).\n  * make_student_sampler -> wraps the returned sampler so each 80 ms chunk\n                           first polls EMOTION_CONTROL_FILE: writing an emotion\n                           id (0-5) or name to that file switches the avatar\'s\n                           emotion mid-conversation. The switch is an in-place\n                           update of the emotion_bias buffer, which the captured\n                           student CUDA graph keeps reading - no re-capture.\n\nEnv vars:\n  EMOTION_LORA_CKPT     adapter checkpoint written by train_emotion_lora.ipynb\n  EMOTION_ID            startup emotion: id 0-5 or name (default 1 = happy)\n  EMOTION_CONTROL_FILE  runtime control file ("" disables runtime switching)\n"""\nimport os\nimport sys\nfrom pathlib import Path\n\nLIVE_DIR = Path(__file__).resolve().parent\nEMOTION_DIR = LIVE_DIR.parents[0] / "emotion"\nfor _p in (str(LIVE_DIR), str(EMOTION_DIR)):\n    if _p not in sys.path:\n        sys.path.insert(0, _p)\n\nimport imtalker_try19_student_live as try19                            # noqa: E402\nfrom emotion_student import (                                          # noqa: E402\n    EMOTION_MAP, EMOTION_NAMES, build_emotion_student)\n\nADAPTER = os.environ.get("EMOTION_LORA_CKPT", "").strip()\nCONTROL = os.environ.get(\n    "EMOTION_CONTROL_FILE",\n    str(LIVE_DIR.parents[2] / "emotion_control.txt")).strip()\n\n\ndef _emotion_index(v):\n    v = str(v).strip().lower()\n    if v.lstrip("-").isdigit():\n        i = int(v)\n        return i if 0 <= i < len(EMOTION_NAMES) else None\n    return EMOTION_MAP.get(v)\n\n\nSTARTUP = _emotion_index(os.environ.get("EMOTION_ID", "1"))\nif STARTUP is None:\n    STARTUP = 1\n\ndef _load_emotion_student(ckpt, device):\n    """try19.load_student replacement: base student + emotion adapter."""\n    if not ADAPTER or not os.path.exists(ADAPTER):\n        raise SystemExit(\n            "[emotion] EMOTION_LORA_CKPT is not set or the file is missing: %r\\n"\n            "[emotion] train an adapter with train_emotion_lora.ipynb first "\n            "(or set EMOTION_HF_REPO in emotion_live_streaming.ipynb)" % ADAPTER)\n    model, cfg, meta = build_emotion_student(ckpt, ADAPTER, device)\n    model.set_emotion(STARTUP)\n    print("[emotion] adapter: %s (trained step %s, LoRA rank %s)"\n          % (ADAPTER, meta.get("step"), meta.get("lora_rank")), flush=True)\n    print("[emotion] startup emotion: %d (%s); runtime control file: %s"\n          % (STARTUP, EMOTION_NAMES[STARTUP], CONTROL or "(disabled)"), flush=True)\n    return model, cfg\n\n\ntry19.load_student = _load_emotion_student\n\n_orig_make_student_sampler = try19.make_student_sampler\n\n\ndef _make_emotion_sampler(model, cfg, **kw):\n    """Wrap try19\'s sampler: poll the control file once per chunk, then run the\n    unmodified student sampler (KV state, blending, gain, CUDA graph)."""\n    inner = _orig_make_student_sampler(model, cfg, **kw)\n    state = {"mtime": None, "idx": None}\n\n    def _poll():\n        if not CONTROL:\n            return\n        try:\n            mtime = os.path.getmtime(CONTROL)\n        except OSError:\n            return\n        if mtime == state["mtime"]:\n            return\n        state["mtime"] = mtime\n        try:\n            idx = _emotion_index(open(CONTROL).read())\n        except Exception:\n            return\n        if idx is None or idx == state["idx"]:\n            return\n        state["idx"] = idx\n        model.set_emotion(idx)          # in-place buffer update: graph-safe\n        print("[emotion] switched to %d (%s)" % (idx, EMOTION_NAMES[idx]),\n              flush=True)\n\n    def sampler(self, helium, target_frames):\n        _poll()\n        return inner(self, helium, target_frames)\n\n    return sampler\n\n\ntry19.make_student_sampler = _make_emotion_sampler\n\nif __name__ == "__main__":\n    try19.main()\n'
path = os.path.join(REPO_DIR, "src", "niloys", "live", "imtalker_emotion_student_live.py")
os.makedirs(os.path.dirname(path), exist_ok=True)
with open(path, "w") as f:
    f.write(CONTENT)
print("wrote", path, "(%d chars)" % len(CONTENT))

## 4 · The launch script

`run_emotion_x86.sh` — a byte-for-byte mirror of `run_x86.sh` (same environment, same GPU
probe, same server flags) that launches the emotion wrapper instead, exports the
`EMOTION_*` variables, and defaults to port 8992.

In [ ]:
import os
CONTENT = '#!/usr/bin/env bash\n# run_emotion_x86.sh -- emotion-aware twin of run_x86.sh. Identical environment\n# and arguments; the only differences:\n#   * launches src/niloys/live/imtalker_emotion_student_live.py (which reuses\n#     imtalker_try19_student_live.py and adds the emotion adapter)\n#   * the EMOTION_* variables below\n#\n#   EMOTION_LORA_CKPT=/path/to/last.pt ./run_emotion_x86.sh\n#   EMOTION_ID=angry PORT=8992 EMOTION_LORA_CKPT=... ./run_emotion_x86.sh\nset -u\nHERE="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"\nSRC="$HERE/src"; W="$HERE/weights"\n[ -s "$HERE/.snap_path" ] && [ -s "$HERE/.python_path" ] || { echo "run ./prepare_x86.sh <hf_token> first"; exit 1; }\nPY="${PY:-$(cat "$HERE/.python_path")}"\nSNAP="$(cat "$HERE/.snap_path")"\nPORT="${PORT:-8992}"   # 8998 is refused by the server; 8991 may hold the base server\nexport HF_HOME="${HF_HOME:-$HERE/hf}"\n\nif [ "${HF_HUB_ENABLE_HF_TRANSFER:-0}" != "0" ] && ! "$PY" -c \'import hf_transfer\' 2>/dev/null; then\n  echo "    note: HF_HUB_ENABLE_HF_TRANSFER=1 but hf_transfer is not installed -> disabling it"\n  export HF_HUB_ENABLE_HF_TRANSFER=0\nfi\nexport TRITON_CACHE_DIR="${TRITON_CACHE_DIR:-$HERE/.cache/triton}" TORCHINDUCTOR_CACHE_DIR="${TORCHINDUCTOR_CACHE_DIR:-$HERE/.cache/inductor}"\nexport DESIGN_A_DIR="$SRC/niloys" IMTALKER_DIR="$SRC/imtalker"\nexport PYTHONPATH="$SRC/niloys:$SRC/moshi_pkg:${PYTHONPATH:-}"\nunset TRY19_DEC_CKPT\nexport NATIVE_PASSTHROUGH=1\nexport TRY19_STUDENT_GRAPH="${TRY19_STUDENT_GRAPH:-1}" TRY19_RENDER_FP16="${TRY19_RENDER_FP16:-1}" TRY19_PD_REFINE="${TRY19_PD_REFINE:-0}"\nexport TRY19_PD="$W/pd_quant_onpol_robert5_plaus2.pt"\nexport TRY19_MIMI_DEC_FT="$W/mimi_dec_gan_robert5.pt"\nexport TRY19_MIMI_GRAPH="${TRY19_MIMI_GRAPH:-1}" TRY19_MIMI_PARTS="${TRY19_MIMI_PARTS:-seanet_quant}" TRY19_MIMI_CAPTURE_AT_INIT="${TRY19_MIMI_CAPTURE_AT_INIT:-1}"\nexport TRY19_CODE_GUARD="${TRY19_CODE_GUARD:-1}"\n\n_int4_user="${TRY19_INT4_G:-}"; _gate_user="${TRY19_DEP_GATING_INT4:-}"\nif [ -s "$HERE/.gpu_probe.env" ]; then set -a; . "$HERE/.gpu_probe.env"; set +a; fi\nexport TRY19_INT4_G="${_int4_user:-${TRY19_INT4_G:-32}}"\nexport TRY19_DEP_GATING_INT4="${_gate_user:-${TRY19_DEP_GATING_INT4:-1}}"\n\nif [ -s "$W/seanet_w12_fp32.engine" ]; then\n  export TRY19_TRT_SEANET="$W/seanet_w12_fp32.engine"\nelse\n  unset TRY19_TRT_SEANET\n  echo "note: no weights/seanet_w12_fp32.engine -- using Mimi\'s PyTorch SEANet decoder"\nfi\n\n# ---- emotion ----\n: "${EMOTION_LORA_CKPT:?set EMOTION_LORA_CKPT to the adapter checkpoint (train_emotion_lora.ipynb)}"\nexport EMOTION_LORA_CKPT\nexport EMOTION_ID="${EMOTION_ID:-1}"\nexport EMOTION_CONTROL_FILE="${EMOTION_CONTROL_FILE:-$HERE/emotion_control.txt}"\necho "$EMOTION_ID" > "$EMOTION_CONTROL_FILE"\n\nSTUDENT_CKPT="${STUDENT_CKPT:-$W/motionM.pt}"\n[ -s "$STUDENT_CKPT" ] || STUDENT_CKPT="$W/motion_student_38M.pt"\n\nmkdir -p "$SRC/niloys/live/static" "$HERE/captures"\nln -sfn "$SRC/imtalker/static/assets" "$SRC/niloys/live/static/assets"\nln -sfn "$SRC/niloys/static/index_try21_no_idle_dissolve.html" "$SRC/niloys/live/static/index.html"\necho "x86 EMOTION: id=$EMOTION_ID adapter=$EMOTION_LORA_CKPT port=$PORT student=$(basename "$STUDENT_CKPT")"\nexec "$PY" -u "$SRC/niloys/live/imtalker_emotion_student_live.py" \\\n  --live_module imtalker_native_passthrough \\\n  --student_ckpt "$STUDENT_CKPT" --distilled_renderer "$W/renderer_narrow.pt" \\\n  --motion_gain "${GAIN:-1.2}" --dep_q 8 \\\n  --blend_frames 0 --jump_thresh 0.15 --mouth_close 0.5 \\\n  --host 0.0.0.0 --port "$PORT" \\\n  --html_path "$SRC/niloys/static/index_try21_no_idle_dissolve.html" \\\n  --generator_path "$W/generator_last.ckpt" --renderer_path "$W/renderer.ckpt" \\\n  --adapter_path "$W/adapter_last.pt" \\\n  --adapter_type unitalk_last_layer --adapter_num_layers 12 --adapter_dropout 0.0 \\\n  --adapter_window_mode lookahead --adapter_future_steps 0 \\\n  --ref_path "$W/ref.jpeg" --wav2vec_model_path "$W/wav2vec2-base-960h" \\\n  --moshi_root "$W" --mimi_hf_repo nvidia/personaplex-7b-v1 \\\n  --moshi_weight "$SNAP/model.safetensors" --mimi_weight "$W/mimi.safetensors" \\\n  --tokenizer "$W/tokenizer_spm_32k_3.model" \\\n  --text_prompt "${TEXT_PROMPT:-You are Robert, an assistant from RB Labs. You have knowledge about crypto, finance, investment, technology, and general topics. Answer every question completely, descriptively, and with useful detail.}" \\\n  --voice_prompt "${VOICE:-Robert_5.pt}" --voice_prompt_dir "$W/voices" \\\n  --enable_moshi_reply --direct_reply_hidden --reply_hidden_steps_per_chunk 1 \\\n  --audio_chunk_sec 2.0 --wav2vec_sec 2.0 --fm_chunk_frames 2 --helium_deque_size 25 \\\n  --prebuffer_chunks "${PREBUFFER:-3}" --render_sub_batch 2 --renderer_precision fp32 \\\n  --frame_q_backpressure 3 --buffer_ms 80 --skip_fm_audio_encoder \\\n  --assistant_speech_rms_threshold 0.006 --assistant_speech_hold_chunks 1 \\\n  --motion_ref_blend 0.0 --motion_prior_noise_blend 0.0 \\\n  --a_cfg_scale 1.24 --nfe 3 --seed 42 --noise_seed 42 --shared_noise \\\n  --fp32 --tf32 \\\n  --silence_helium_path "$W/silence_helium_mean.pt" \\\n  --jpeg_quality 90 --device cuda --reply_audio_gain 1.0 --output_audio_codec opus \\\n  --capture_dir "$HERE/captures" \\\n  --blink_motion_path "$W/blink_motion.pt"\n'
path = os.path.join(REPO_DIR, "run_emotion_x86.sh")
os.makedirs(os.path.dirname(path), exist_ok=True)
with open(path, "w") as f:
    f.write(CONTENT)
print("wrote", path, "(%d chars)" % len(CONTENT))
sh("chmod +x %s" % path)

## 5 · Start the emotion server

Detached with `setsid` (survives kernel restarts), output to `emotion_run.log`. First
start takes a few minutes (trunk load + int4 conversion + CUDA graph captures); later
starts reuse the kernel caches. The cell waits until `/health` reports `loaded` and then
shows the emotion lines from the log.

In [ ]:
sh("PORT=%d VOICE=%s TEXT_PROMPT=%s GAIN=%s PREBUFFER=%s EMOTION_ID=%s EMOTION_LORA_CKPT=%s "
   "nohup setsid ./run_emotion_x86.sh > emotion_run.log 2>&1 < /dev/null & "
   "sleep 3; echo started:; pgrep -af imtalker_emotion_student_live.py | head -3"
   % (PORT, VOICE, json.dumps(TEXT_PROMPT), GAIN, PREBUFFER,
      shlex.quote(EMOTION), shlex.quote(EMOTION_LORA_CKPT)))


def wait_loaded(port, timeout=1800):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            with urllib.request.urlopen("http://127.0.0.1:%d/health" % port, timeout=5) as r:
                h = json.load(r)
            if h.get("loaded"):
                print("\nloaded after %.0fs: %s" % (time.time() - t0, h))
                return True
        except Exception:
            pass
        if subprocess.run(["pgrep", "-f", "imtalker_emotion_student_live.py"],
                          capture_output=True).returncode != 0:
            print("\n!! the server process is gone - last 40 lines of emotion_run.log:")
            sh("tail -40 emotion_run.log", check=False)
            return False
        tail = subprocess.run(["bash", "-c", "tail -1 emotion_run.log"],
                              capture_output=True, text=True, cwd=REPO_DIR).stdout.strip()
        print("  ... %5.0fs  %s" % (time.time() - t0, tail[:110]), flush=True)
        time.sleep(15)
    print("\n!! timed out")
    sh("tail -40 emotion_run.log", check=False)
    return False


ok = wait_loaded(PORT)
print()
sh("grep -aE '\\[emotion\\]|x86 EMOTION|int4 G=|Uvicorn running' emotion_run.log | head -20",
   check=False)

## 6 · Open it in Chrome

The microphone needs `https://` (or `http://localhost`). Option A is the RunPod HTTP proxy
(if the port is in the pod's HTTP ports); option B is a Cloudflare quick tunnel (no pod
configuration).

In [ ]:
pod = os.environ.get("RUNPOD_POD_ID", "")
if pod:
    print("  https://%s-%d.proxy.runpod.net/\n" % (pod, PORT))
    print("If that 502s, port %d is not in this pod's HTTP ports - use the tunnel below." % PORT)
else:
    print("RUNPOD_POD_ID is not set - use the tunnel below, or SSH:")
    print("  ssh -L %d:localhost:%d root@<pod-host> -p <ssh-port>   then http://localhost:%d/"
          % (PORT, PORT, PORT))

In [ ]:
import re
os.makedirs(os.path.join(REPO_DIR, ".bin"), exist_ok=True)
CF = os.path.join(REPO_DIR, ".bin", "cloudflared")
if not os.path.exists(CF):
    print("downloading cloudflared ...")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CF)
    os.chmod(CF, 0o755)

sh("pkill -f 'cloudflared tunnel'; sleep 1", check=False)
tun = subprocess.Popen([CF, "tunnel", "--no-autoupdate", "--url",
                        "http://localhost:%d" % PORT],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                       text=True, bufsize=1)
public, t0 = None, time.time()
while time.time() - t0 < 90:
    line = tun.stdout.readline()
    if not line:
        break
    m = re.search(r"https://[-\w.]+trycloudflare\.com", line)
    if m:
        public = m.group(0)
        break
if public:
    print("\n" + "=" * 72)
    print("  OPEN THIS IN CHROME:   %s/" % public)
    print("=" * 72)
    print("\nClick 'Start conversation', allow the microphone, and speak.")
else:
    print("could not get a tunnel URL; last output:")
    print(tun.stdout.read()[-2000:] if tun.stdout else "")

## 7 · Headless end-to-end check (no browser, no microphone)

Streams 20 s of recorded speech into the server, then silence. `PASS` = it heard the
question, answered with speech, and streamed video above 15 fps. The avatar's motion in
this run uses the **startup emotion**.

In [ ]:
sh("%s tools/test_session.py %d 40" % (shlex.quote(PY), PORT), check=False)

## 8 · Switch emotion mid-conversation

The wrapper polls `emotion_control.txt` once per 80 ms chunk; writing a new value applies
on the next chunk — no restart, no re-capture, while a conversation is running. Valid
values: `0`–`5` or `neutral`, `happy`, `sad`, `angry`, `surprise`, `fear`.

In [ ]:
EMOTION_NAMES = ["neutral", "happy", "sad", "angry", "surprise", "fear"]
CONTROL = os.path.join(REPO_DIR, "emotion_control.txt")

def set_emotion(name_or_id):
    v = str(name_or_id).strip().lower()
    if not v.isdigit():
        v = str(EMOTION_NAMES.index(v))
    assert 0 <= int(v) < 6
    with open(CONTROL, "w") as f:
        f.write(v)
    print("emotion -> %s (%s) - applies on the next 80 ms chunk"
          % (v, EMOTION_NAMES[int(v)]))

set_emotion(EMOTION)   # try: set_emotion("angry"), set_emotion("sad"), ...

In [ ]:
# watch the switch land in the server log
sh("sleep 1; grep -a '\\[emotion\\]' emotion_run.log | tail -5", check=False)

## 9 · Day-to-day: logs, restart, stop

In [ ]:
sh("tail -60 emotion_run.log", check=False)

In [ ]:
# step times and anything worth knowing about
sh("grep -aE 'step time|over budget|WARNING|Error|Traceback' emotion_run.log | tail -40",
   check=False)

In [ ]:
# restart with a different startup emotion (stops the current server first)
EMOTION = "angry"
sh("pkill -f imtalker_emotion_student_live.py; sleep 2", check=False)
sh("PORT=%d VOICE=%s TEXT_PROMPT=%s GAIN=%s PREBUFFER=%s EMOTION_ID=%s EMOTION_LORA_CKPT=%s "
   "nohup setsid ./run_emotion_x86.sh > emotion_run.log 2>&1 < /dev/null & "
   "sleep 3; pgrep -af imtalker_emotion_student_live.py | head -3"
   % (PORT, VOICE, json.dumps(TEXT_PROMPT), GAIN, PREBUFFER,
      shlex.quote(EMOTION), shlex.quote(EMOTION_LORA_CKPT)))
# then re-run the wait cell (section 5)

In [ ]:
# stop everything
sh("pkill -f imtalker_emotion_student_live.py; pkill -f 'cloudflared tunnel'; sleep 2; "
   "nvidia-smi --query-gpu=memory.used --format=csv,noheader", check=False)

### Troubleshooting

| symptom | fix |
|---|---|
| `[emotion] EMOTION_LORA_CKPT is not set or the file is missing` | train the adapter (`train_emotion_lora.ipynb`) or set `EMOTION_HF_REPO` and re-run preflight |
| `CUDA out of memory` while loading | the base server is still running — the preflight stops it; check `nvidia-smi` |
| emotion switches do nothing | `grep -a '\[emotion\]' emotion_run.log` — the control-file path printed at startup must be `emotion_control.txt` in the repo root |
| avatar looks identical across emotions | the adapter biases may be near zero — see the validation cell of the training notebook |
| microphone blocked | the page must be `https://` or `http://localhost` — use the tunnel |
| everything else | the base pipeline's troubleshooting table in `runpod_speech2face.ipynb` applies verbatim (same server machinery) |